In [1]:
import os
from pathlib import Path

import pandas as pd
from dotenv import load_dotenv
from google.analytics.data_v1beta import BetaAnalyticsDataClient
from google.analytics.data_v1beta.types import (
    DateRange,
    Dimension,
    Metric,
    RunReportRequest,
)

In [2]:
load_dotenv()
PROPERTY_ID = os.getenv("GA4_PROPERTY_ID")
KEY_FILE_LOCATION = os.getenv("GA4_CREDENTIALS_PATH")
RAW_OUTPUT_DIR = Path("outputs/raw")
OUTPUT_DIR = RAW_OUTPUT_DIR
SUPPORTED_METRICS = (
    "activeUsers",
    "newUsers",
    "sessions",
    "engagedSessions",
    "eventCount",
    "userEngagementDuration",
)

if not PROPERTY_ID:
    raise ValueError("GA4_PROPERTY_ID is not set in the environment.")

if not KEY_FILE_LOCATION:
    raise ValueError("GA4_KEY_FILE_LOCATION is not set in the environment.")

## User Demographic


In [3]:
GROUPED_DEMOGRAPHIC_DIMENSIONS = {
    "group1": ["userAgeBracket", "userGender", "language"],
    "group2": ["country", "region", "city"],
}


def get_client():
    """Builds a GA4 Data API client from the service account JSON file."""
    key_file = Path(KEY_FILE_LOCATION)
    if not key_file.exists():
        raise FileNotFoundError(f"GA4 key file was not found: {key_file}")

    return BetaAnalyticsDataClient.from_service_account_json(filename=str(key_file))


def fetch_demographic_report(
    property_id,
    dimension_names,
    metrics=SUPPORTED_METRICS,
    start_date="28daysAgo",
    end_date="today",
):
    """Fetches a GA4 demographic report grouped by yearMonth and one or more demographic dimensions."""
    dimension_names = (
        [dimension_names] if isinstance(dimension_names, str) else list(dimension_names)
    )

    client = get_client()
    request = RunReportRequest(
        property=f"properties/{property_id}",
        dimensions=[Dimension(name="yearMonth")]
        + [Dimension(name=name) for name in dimension_names],
        metrics=[Metric(name=metric_name) for metric_name in metrics],
        date_ranges=[DateRange(start_date=start_date, end_date=end_date)],
    )
    return client.run_report(request)


def response_to_dataframe(response, dimension_names, start_date, end_date):
    """Converts a GA4 run_report response into a tidy dataframe."""
    dimension_names = (
        [dimension_names] if isinstance(dimension_names, str) else list(dimension_names)
    )
    metric_names = [header.name for header in response.metric_headers]
    rows = []

    for row in response.rows:
        row_data = {
            "demographic_dimensions": ",".join(dimension_names),
            "date_range_start": start_date,
            "date_range_end": end_date,
        }

        for header, dim_value in zip(response.dimension_headers, row.dimension_values):
            if header.name == "yearMonth":
                row_data["year_month"] = dim_value.value
            else:
                row_data[header.name] = dim_value.value

        for metric_name, metric_value in zip(metric_names, row.metric_values):
            value = metric_value.value
            if value.replace(".", "", 1).isdigit():
                row_data[metric_name] = float(value) if "." in value else int(value)
            else:
                row_data[metric_name] = value

        rows.append(row_data)

    dataframe = pd.DataFrame(rows)

    preferred_order = [
        "demographic_dimensions",
        "year_month",
        *dimension_names,
        *metric_names,
        "date_range_start",
        "date_range_end",
    ]
    ordered_columns = [
        column for column in preferred_order if column in dataframe.columns
    ]
    remaining_columns = [
        column for column in dataframe.columns if column not in ordered_columns
    ]
    dataframe = dataframe[ordered_columns + remaining_columns]

    if not dataframe.empty:
        sort_columns = ["year_month"] + [
            name for name in dimension_names if name in dataframe.columns
        ]
        dataframe = dataframe.sort_values(sort_columns).reset_index(drop=True)

    return dataframe


def export_demographic_report(
    property_id,
    dimension_names,
    metrics=SUPPORTED_METRICS,
    output_dir=OUTPUT_DIR,
    output_file_name=None,
    start_date="7daysAgo",
    end_date="today",
):
    """Exports one demographic report to a CSV file."""
    response = fetch_demographic_report(
        property_id,
        dimension_names,
        metrics=metrics,
        start_date=start_date,
        end_date=end_date,
    )
    dataframe = response_to_dataframe(response, dimension_names, start_date, end_date)

    output_dir.mkdir(parents=True, exist_ok=True)
    dimension_names_list = (
        [dimension_names] if isinstance(dimension_names, str) else list(dimension_names)
    )
    if output_file_name is None:
        output_file_name = f"ga4_demographics_{'_'.join(dimension_names_list)}.csv"
    output_path = output_dir / output_file_name
    dataframe.to_csv(output_path, index=False)

    name_label = (
        dimension_names
        if isinstance(dimension_names, str)
        else ",".join(dimension_names_list)
    )
    if dataframe.empty:
        print(f"{name_label}: no rows returned; wrote empty CSV to {output_path}")
    else:
        print(f"{name_label}: wrote {len(dataframe)} rows to {output_path}")

    return output_path


def export_demographic_reports(
    property_id=PROPERTY_ID,
    dimension_groups=GROUPED_DEMOGRAPHIC_DIMENSIONS,
    metrics=SUPPORTED_METRICS,
    output_dir=OUTPUT_DIR,
    start_date="28daysAgo",
    end_date="today",
):
    """Exports all grouped demographic reports to separate CSV files."""
    output_paths = []
    output_file_names = {
        "group1": "ga4_users_demographic.csv",
        "group2": "ga4_users_location.csv",
    }
    for group_name, dimension_names in dimension_groups.items():
        output_paths.append(
            export_demographic_report(
                property_id=property_id,
                dimension_names=dimension_names,
                metrics=metrics,
                output_dir=output_dir,
                output_file_name=output_file_names.get(group_name),
                start_date=start_date,
                end_date=end_date,
            )
        )

    return output_paths

In [12]:
def transform_demographic_csv(input_path, output_dir):
    """Reads a demographic CSV, drops helper columns, and splits year_month into year and month."""
    dataframe = pd.read_csv(input_path)
    drop_columns = [
        "demographic_dimensions",
        "date_range_start",
        "date_range_end",
    ]
    dataframe = dataframe.drop(
        columns=[col for col in drop_columns if col in dataframe.columns]
    )

    if "year_month" in dataframe.columns:
        dataframe["year_month"] = dataframe["year_month"].astype(str).str.zfill(6)
        dataframe["year"] = dataframe["year_month"].str[:4]
        dataframe["month"] = dataframe["year_month"].str[4:6]
        dataframe = dataframe.drop(columns=["year_month"])

        remaining_columns = [
            col for col in dataframe.columns if col not in ["year", "month"]
        ]
        dataframe = dataframe[["year", "month", *remaining_columns]]

    output_dir.mkdir(parents=True, exist_ok=True)
    output_path = output_dir / input_path.name
    dataframe.to_csv(output_path, index=False)
    return output_path


def transform_grouped_demographic_reports(
    input_dir=OUTPUT_DIR,
    output_dir=Path("outputs/transform"),
    filenames=("ga4_users_demographic.csv", "ga4_users_location.csv"),
):
    """Transforms grouped demographic report CSVs and writes them to outputs/transform."""
    output_paths = []
    for filename in filenames:
        input_path = input_dir / filename
        if not input_path.exists():
            raise FileNotFoundError(f"Expected input CSV not found: {input_path}")

        output_paths.append(transform_demographic_csv(input_path, output_dir))

    return output_paths

In [5]:
# Export and store 2 years of demographic history
export_demographic_reports(
    PROPERTY_ID,
    start_date="2026-01-01",
    end_date="yesterday",
)

userAgeBracket,userGender,language: wrote 23 rows to outputs/raw/ga4_users_demographic.csv
country,region,city: wrote 516 rows to outputs/raw/ga4_users_location.csv


[PosixPath('outputs/raw/ga4_users_demographic.csv'),
 PosixPath('outputs/raw/ga4_users_location.csv')]

## Transform


In [13]:
transform_grouped_demographic_reports()

[PosixPath('outputs/transform/ga4_users_demographic.csv'),
 PosixPath('outputs/transform/ga4_users_location.csv')]

# Load to Supabase